In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from tueplots import bundles, cycler, figsizes
from tueplots.constants.color import palettes

plt.rcParams.update(bundles.icml2022())
plt.rcParams.update(figsizes.icml2022_full())
plt.rcParams.update(cycler.cycler(color=palettes.tue_plot))
plt.rcParams.update({"figure.dpi": 350})

In [ ]:
path = "../../data/newspaper_collection_evaluation_results_20_12_2025.csv"
np_coll_df = pd.read_csv(path, index_col=False)
np_coll_df["date"] = pd.to_datetime(np_coll_df["date"])
# emotions_str = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
emotions_str = ['angry','fear', 'happy', 'sad', 'surprise', 'neutral', "surprise"]

### Available Attributes:
- **personal attributes**
    - name, 
    - surname, 
    - fullname, 
    - birth, 
    - gender, 
- **additional info**
    - article, 
    - party, 
    - newspaper, 
- **evaluation metadata**
    - confidence, 
    - distance, 
    - date, 
- **emotions**:
    - dominant_emotion, 
    - angry, 
    - disgust, 
    - fear, 
    - happy, 
    - sad, 
    - surprise, 
    - neutral, 

In [ ]:
np_coll_df["newspaper"].unique()

In [ ]:
newspaper_name_mapping = {
    "sz": "sz",
    "stern": "stern",
    "freitag": "frtg",
    "taz": "taz",
    "compact": "cmpct",
    "spiegel": "spgl",
    "nd": "nd"
}
np_coll_df["newspaper"] = np_coll_df["newspaper"].map(newspaper_name_mapping)

In [ ]:
party_name_mapping = {
    "afd": "adf",
    "transnational": "trnsntl",
    "gruenen": "gruene",
    "spd": "spd",
    "linke": "linke",
    "union": "union"
}

np_coll_df["party"] = np_coll_df["party"].map(party_name_mapping)

## Chose a confidence to work with

In [ ]:
min_confidence = 70
cut_off_confidence_df = np_coll_df.query("confidence > @min_confidence")
confidence_mean = cut_off_confidence_df["confidence"].mean()
print("*================================================*")
print(f"| {len(cut_off_confidence_df)}/{len(np_coll_df)} Samples remain with confidence > {min_confidence} |")
print(f"|       Resulting confidence mean: {confidence_mean:.2f}        |")
print("*================================================*")
np_coll_df = cut_off_confidence_df

## Drop Rows that make only a small proportion 

In [ ]:
np_coll_df = np_coll_df.query("party != 'fdp'")
np_coll_df = np_coll_df.drop(columns=["disgust"])
np_coll_df.head(1)

In [ ]:
def barplot_factory(data, x, y, hue, ax, alpha=1, width=1, mean_attr=None):
    if mean_attr:
        overall_mean = np_coll_df[mean_attr].mean()

    sns.barplot(data=data, x=x, y=y, hue=hue, alpha=alpha, width=width, ax=ax, dodge=False)
    if mean_attr: ax.axhline(overall_mean, color='darkred', linestyle='--', linewidth=1)
    
    for container in ax.containers:
        if hasattr(container, 'datavalues'):
            ax.bar_label(container, fmt='%.2f', label_type='edge', padding=-15, color='white', fontweight='bold')

In [ ]:
fig, ax = plt.subplots(1, 2)
barplot_factory(data=np_coll_df, x="party", y="confidence", hue="party", alpha=0.9, width=0.7, ax=ax[0], mean_attr="confidence")
barplot_factory(data=np_coll_df, x="newspaper", y="confidence", hue="newspaper", alpha=0.9, width=0.7, ax=ax[1], mean_attr="confidence")
plt.savefig("fig/confidence_distr_party_newspaper_cutoff_{min_confidence}_mean_{}.pdf")
plt.show()

## How is the data distributed?

In [ ]:
# party
fig, ax = plt.subplots(2, 2, figsize=(8, 3))
fig.suptitle("Distribution of Samples by Different Categories")
ax[0, 0].set_title("By Party")
sns.countplot(data=np_coll_df, x="party", hue="party", ax=ax[0, 0], order=np_coll_df["party"].value_counts().index)
ax[0, 1].set_title("By Newspaper")
sns.countplot(data=np_coll_df, x="newspaper", hue="newspaper", ax=ax[0, 1], order=np_coll_df["newspaper"].value_counts().index)
ax[1, 0].set_title("Gender")
sns.countplot(data=np_coll_df, x="gender", hue="gender", ax=ax[1, 0], order=np_coll_df["gender"].value_counts().index)
ax[1, 1].set_title("Dominant Emotion")
sns.countplot(data=np_coll_df, x="dominant_emotion", hue="dominant_emotion", ax=ax[1, 1], order=np_coll_df["dominant_emotion"].value_counts().index)
plt.savefig("fig/sample_distribution_party_gender_newspaper_emotion.pdf")
plt.show()

In [ ]:
# party, emotion, newspaper
grouped_emotions_df = np_coll_df.groupby(by=["newspaper", "party", "date"])[emotions_str].mean()
grouped_emotions_df

In [ ]:
import matplotlib.dates as mdates
plot_cell = False
if plot_cell:
    for e in emotions_str:
        data = grouped_emotions_df.loc[("compact", "afd"), [e]].reset_index()
        df_long = data.melt(id_vars="date", var_name="emotion", value_name="score")
        fig, ax = plt.subplots(figsize=(6, 2))
        sns.lineplot(df_long, x="date", y="score", hue="emotion", ax=ax)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        plt.xticks(rotation=25)
        plt.show()

In [ ]:
rows, cols = 2, 3
fig, ax = plt.subplots(rows, cols, figsize=(5, 3))
for i in range(rows):
    for j in range(cols):
        idx = (i*cols) + j
        c_emotion = emotions_str[idx]
        agg_party_newspaper = np_coll_df.groupby(["party", "newspaper"], as_index=False)[c_emotion].mean()
        ax[i, j].set_title(c_emotion)
        sns.scatterplot(
            data=agg_party_newspaper,
            x="newspaper",
            y="party",
            hue="newspaper",
            size=c_emotion,
            sizes=(1, 100),
            legend=False,
            alpha=0.8,
            marker="8",
            ax=ax[i, j],
            edgecolor="black",
            linewidth=0.5,
        )
        ax[i, j].set_xlabel("")
        ax[i, j].set_ylabel("")
plt.tight_layout()
plt.savefig("fig/mean_emotions_by_party_newspaper.pdf", dpi=300)

In [ ]:
rows, cols = 2, 3
fig, ax = plt.subplots(rows, cols, figsize=(9, 8))

for i in range(rows):
    for j in range(cols):
        idx = (i * cols) + j
        c_emotion = emotions_str[idx]
        agg_party_newspaper = np_coll_df.groupby(["party", "newspaper"], as_index=False)[c_emotion].mean()
        pivot_df = agg_party_newspaper.pivot(index="party", columns="newspaper", values=c_emotion)
        
        sns.heatmap(
            pivot_df, 
            ax=ax[i, j], 
            annot=True, 
            fmt=".2f", 
            cmap="YlGnBu", 
            cbar=False,
            square=True,
            annot_kws={"fontsize": 8}
        )
        ax[i, j].set_title(c_emotion)
        ax[i, j].set_xlabel(None)
        ax[i, j].set_ylabel(None)
plt.savefig("fig/heatmap_emotion_by_party_newspaper.pdf")
plt.tight_layout()

In [ ]:
rows, cols = 2, 3
fig, ax = plt.subplots(rows, cols, figsize=(12, 7))

for i in range(rows):
    for j in range(cols):
        idx = (i * cols) + j
        c_emotion = emotions_str[idx]
        agg_party_newspaper = np_coll_df.groupby(["party", "newspaper"], as_index=False)[c_emotion].mean()
        
        ax[i, j].set_title(c_emotion)
        sns.scatterplot(
            data=agg_party_newspaper,
            x="newspaper",
            y="party",
            hue="newspaper",
            size=c_emotion,
            sizes=(100, 1000),
            legend=False,
            alpha=0.8,
            marker="o",
            ax=ax[i, j],
            edgecolor="black",
            linewidth=1.5
        )
        ax[i, j].set_xlabel("")
        ax[i, j].set_ylabel("")
        ax[i, j].margins(x=0.1, y=0.1)

plt.tight_layout()